# Weekly Themed Meal Planner: refactored

**Lab:** From hacky script to "we could ship this" · **Path 1:** refactor my own course code, the Weekly Themed Meal Planner MVP from class.

Each section below is one step of the lab instructions (Steps 1 to 6), with short explanations between the code cells. Steps 7 and 8 are optional and not part of this submission.

| File | Role |
|---|---|
| `original/meal_planner_starter.ipynb` | untouched copy of the original notebook (Step 1) |
| `data/meal_requests.json` | **input**: the original 4 test cases, now a file |
| `data/malformed.json`, `data/invalid_requests.json` | broken inputs for the error tests |
| `output/` | created on run: the baseline of the original and the **output** record |
| `lab_proof.md` | reviewer entry point; its evidence blocks are filled by the last cells |
| `.env` | your keys, never committed (copy `.env.example`) |

**How to run:** put a `.env` file with `OPENAI_API_KEY` next to this notebook (the email settings are optional), then Run All. Only two cells call the paid API, both marked **Real API call**: the baseline in Step 1 and the workflow run in Step 6.

## 0. Setup

Install only what the code imports. The original also installed `langchain` and `langchain-anthropic`, which nothing used (proved in Step 2).

In [19]:
%pip install langchain-openai langchain-core openai pydantic gradio python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [20]:
import json
import os
import re
import smtplib
import time
from datetime import datetime
from email.message import EmailMessage
from enum import Enum
from pathlib import Path
from typing import List

import gradio as gr
import openai
from dotenv import load_dotenv
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, ValidationError, field_validator

### Settings

The refactored configuration: secrets are read from `.env` and never typed into the notebook, and every number or string the code relies on is a named constant, defined once here. The last lines only report whether each secret exists, never its value.

In [21]:
# Secrets come from .env (the original typed the email password into this cell)
load_dotenv(".env")
EMAIL_SENDER = os.getenv("EMAIL_SENDER")              # optional, only for the Email Plan button
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

# Model and network settings
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
TEMPERATURE = 0.7
TIMEOUT_SECONDS = 60          # OpenAI and Gmail: a hung connection becomes a clear timeout error
API_MAX_RETRIES = 2           # retries built into the OpenAI client (network errors, 429, 5xx)
SMTP_HOST, SMTP_PORT = "smtp.gmail.com", 587
EMAIL_SUBJECT = "Your Weekly Meal Plan"

# Meal plan rules (the original typed 7 and 5 again in the prompt and in run_basic_validation)
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
RECIPES_PER_PLAN = len(DAYS)
INGREDIENTS_PER_RECIPE = 5
EXCLUDED_BASICS = ["salt", "pepper", "oil", "garlic", "water"]
DEFAULT_THEMES = "random diverse cuisines"
DEFAULT_PREFERENCES = "none"

# Workflow files: one input, one output
REQUESTS_FILE = "data/meal_requests.json"
PROOF_REQUEST = 2             # request #2 (vegetarian) uses all three input fields
BASELINE_FILE = "output/baseline_original.json"
OUTPUT_FILE = "output/meal_plan_record.json"

for name in ["OPENAI_API_KEY", "EMAIL_SENDER", "EMAIL_APP_PASSWORD"]:
    print(f"{name:<20} {'set' if os.getenv(name) else 'MISSING'}")

OPENAI_API_KEY       set
EMAIL_SENDER         set
EMAIL_APP_PASSWORD   set


## Step 1: Select your code

**Path 1, my own code:** the Weekly Themed Meal Planner notebook from class. I chose it because it needs more refactoring than a single script: one 40-line method does seven jobs, the email password sits in the config cell, and the Email button quietly creates a second, different plan.

The original is kept unchanged in `original/meal_planner_starter.ipynb`. This notebook only reads it.

In [22]:
# Checkpoint: the preserved copy exists. List its sections to confirm it is the right notebook.
original = json.loads(Path("original/meal_planner_starter.ipynb").read_text(encoding="utf-8"))
original_cells = ["".join(cell["source"]) for cell in original["cells"] if cell["cell_type"] == "code"]

for cell in original["cells"]:
    if cell["cell_type"] == "markdown":
        print("".join(cell["source"]).splitlines()[0])
print(f"\n{len(original_cells)} code cells")

# 🍽️ Weekly Themed Meal Planner MVP
## 1. Installation & Setup
## 2. Configuration
## 3. Pydantic Models for Validation
## 4. LangChain Setup & Prompts
## 5. Meal Generation Logic
## 6. Email Functionality
## 7. Validation
## 8. Gradio Interface
## 9. Launch Application
## 10. Run Test Cases

11 code cells


### Run the original once (baseline) · Real API call

The baseline runs the original code **exactly as written**: its config, env, models, prompts and generator cells, executed in a separate namespace so none of its names mix with the refactored code. Skipped on purpose: `%pip`, the Gradio launch (it opens a public `share=True` link) and the test loop (4 more paid calls).

The input is request #2 from `data/meal_requests.json`, the same payload the refactored workflow uses in Step 6, so the two runs can be compared.

In [23]:
# Run the ORIGINAL code unchanged and record what happens (success or crash) as the baseline
baseline_input = json.loads(Path(REQUESTS_FILE).read_text(encoding="utf-8"))["requests"][PROOF_REQUEST - 1]
namespace = {"__name__": "original_notebook"}

try:
    for label, code in zip(["config", "env", "models", "prompts", "generator"], original_cells[1:6]):
        print("running original cell:", label)
        exec(code, namespace)
    baseline = namespace["generator"].generate(**baseline_input)
except Exception as e:   # deliberately broad: here we only OBSERVE how the original fails
    baseline = {"success": False, "meal_plan": None, "error": f"{type(e).__name__}: {e}"}

Path(BASELINE_FILE).parent.mkdir(exist_ok=True)
Path(BASELINE_FILE).write_text(json.dumps(baseline, indent=2), encoding="utf-8")
print("\nInput:", baseline_input)
if baseline["success"]:
    print(f"Baseline OK: {len(baseline['meal_plan']['recipes'])} recipes in {baseline['generation_time']}s, saved to {BASELINE_FILE}")
else:
    print("Baseline FAILED:", baseline["error"])

running original cell: config
running original cell: env
running original cell: models
running original cell: prompts
running original cell: generator

Input: {'dietary_preference': 'vegetarian', 'theme_requests': 'Italian, Asian, Mexican', 'additional_preferences': 'no mushrooms'}
Baseline FAILED: OpenAIAuthenticationError: Error code: 401 - {'error': {'message': 'Your API key has been invalidated.', 'type': None, 'code': 'token_invalidated', 'param': None}, 'status': 401}


### What the original does when something is wrong

| Check (from the hints) | What the original does |
|---|---|
| `.env` missing | the "Set environment variables" cell, `os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY`, crashes with `TypeError: str expected, not NoneType` (reproduced below). The message never says that the key is missing. |
| Values typed into the config cell | `EMAIL_SENDER` and `EMAIL_PASSWORD` are plain text in the notebook, so they would go to GitHub with it. |
| Email Plan with placeholder settings | it first generates a brand-new plan (a second paid call, different recipes), then the Gmail login fails with `Failed to send email: ...` and no hint that the settings are placeholders. |
| Install line | `langchain` and `langchain-anthropic` are installed but never imported (checked in Step 2). |

In [24]:
# Reproduce the original env cell without a key (os.getenv returned None), without touching the real .env
try:
    os.environ["OPENAI_API_KEY_DEMO"] = None
except TypeError as e:
    print(f"Original behaviour when .env is missing -> {type(e).__name__}: {e}")

Original behaviour when .env is missing -> TypeError: str expected, not NoneType


## Step 2: Analyze the code

Before changing a line, I read the original and backed the findings with quick counts from its source code.

In [25]:
# Evidence: repeated and hardcoded code in the ORIGINAL source
source = "\n".join(original_cells)
patterns = {
    "ingredient line built by hand (email + UI)": "for ing in recipe['ingredients']",
    "empty-textbox default written by hand": "theme_requests if theme_requests",
    "generator.generate( calls (Generate, Email, tests)": "generator.generate(",
    "recipe count typed as 7": "!= 7",
    "ingredient count typed as 5": "!= 5",
}
for label, pattern in patterns.items():
    print(f"{source.count(pattern)}x  {label}")

2x  ingredient line built by hand (email + UI)
3x  empty-textbox default written by hand
3x  generator.generate( calls (Generate, Email, tests)
2x  recipe count typed as 7
1x  ingredient count typed as 5


In [26]:
# Evidence: dead code (defined but never used) and installed-but-unused packages
for name in ["SHOPPING_LIST_PROMPT", "shopping_parser", "EXCLUDED_BASICS", "DAYS", "DietaryPreference", "Optional"]:
    print(f"{name:<22} used {source.count(name) - 1}x after its definition")

print()
for package in original_cells[0].split("install -q")[1].split():
    module = "dotenv" if package == "python-dotenv" else package.replace("-", "_")
    imported = re.search(rf"^\s*(from|import) {module}\b", source, re.MULTILINE)
    print(f"{package:<22} {'imported' if imported else 'NOT USED'}")

SHOPPING_LIST_PROMPT   used 0x after its definition
shopping_parser        used 0x after its definition
EXCLUDED_BASICS        used 0x after its definition
DAYS                   used 0x after its definition
DietaryPreference      used 0x after its definition
Optional               used 0x after its definition

langchain              NOT USED
langchain-openai       imported
langchain-anthropic    NOT USED
pydantic               imported
gradio                 imported
python-dotenv          imported


## Refactoring Checklist

### Issues Found:
- [x] Function `generate()` does too much (builds the chain AND fills defaults AND calls the API AND strips fences AND parses JSON AND validates AND times the run) → Step 4
- [x] Error handling missing in function `generate()`: `recipe_chain.invoke()` sits outside the `try`, so a wrong key or no network crashes the app → Steps 4, 5
- [x] Validation errors caught but not shown clearly: `except Exception` returns `str(e)`, which says what broke but not where, or how to fix it → Steps 3, 5
- [x] Hardcoded secrets in the config cell (`EMAIL_SENDER`, `EMAIL_PASSWORD`), and a missing `.env` crashes the env cell with `TypeError: str expected, not NoneType` → Setup
- [x] Bug in `send_email_ui()`: it calls `generate()` again, so the email holds a different plan than the screen and costs a second API call → Step 4
- [x] Code repeated in functions `format_meal_plan_email` and `format_single_recipe` (the ingredient line, could be a helper function) → Step 3
- [x] Code repeated in functions `generate_plan_streaming` and `send_email_ui` (textbox defaults; a third copy in `generate()` uses a different default, and `None` crashes `.strip()`) → Step 3
- [x] Fragile fence stripping in `generate()`: unfenced JSON with a sentence before it, or an upper-case ```` ```JSON ```` fence, breaks parsing → Step 3
- [x] Inconsistent output in `generate()`: the success dict has `generation_time`, the failure dicts do not → Step 3
- [x] Duplicated validation: `run_basic_validation()` re-checks the 7 recipes, unique themes and 5 ingredients rules the Pydantic models already enforce, with `7` and `5` typed again → Step 4
- [x] Hardcoded prompt in `MEAL_PLAN_PROMPT`: 5, 7 and the basics list are typed into the text → Step 4
- [x] Mixed concerns in `send_meal_plan_email()`: builds the message AND logs in AND sends, with the SMTP server hardcoded → Step 4
- [x] Dead code: `SHOPPING_LIST_PROMPT`, `shopping_parser`, `EXCLUDED_BASICS`, `DAYS`, `DietaryPreference`, `Optional`, plus `langchain` and `langchain-anthropic` installed but never imported → removed or now used
- [x] No error message when file not found: there is no input file at all; the test cases sit in an inline list, make 4 real API calls and stop at the first crash → Step 6
- [x] `app.launch(share=True)` publishes a public link to an app that spends my API credit and can send email → app section

### Priority:
1. Secrets and exposure: the email password typed into the notebook, and the public share link
2. Silent or unlocated failures: the unprotected API call and `str(e)` messages that never say where
3. The email bug: the emailed plan is not the plan on screen, and it costs a second API call

## Step 3: Create helper functions

Every piece of logic that appeared more than once now lives in one small function, and each one can be tested with hand-made data, without calling OpenAI.

| Helper | Replaces in the original |
|---|---|
| `make_error()` | `except Exception: return str(e)`: errors said *what* broke, never *where* or how to fix it. Now there is one format for every error. |
| `text_or_default()` | the "empty textbox → default" rule, written 3 times with 2 different defaults, crashing on `None` |
| `format_ingredients()` | the ingredient line copied into the email and the UI |
| `extract_json_text()` | the two-branch ```` ``` ```` fence split that breaks on a sentence before unfenced JSON or on an upper-case ```` ```JSON ```` fence |
| `parse_json_text()` | JSON parsing, now shared by the input file and the model answer, with line and column |
| `make_result()` | success and failure dicts with different keys |
| `trace()` | (new) one line of the execution trace for `lab_proof.md` |

In [27]:
class MealPlannerError(Exception):
    # Raised by every refactored function; the message already says where it happened and what to do
    pass


def make_error(function_name, error, location, suggestion):
    # Helper: the ONE error format: function, error type, message, location, suggestion
    if isinstance(error, ValidationError):   # list each invalid field instead of Pydantic's long dump
        message = f"{error.error_count()} invalid field(s)" + "".join(
            f"\n              - {'.'.join(map(str, err['loc'])) or 'input'}: {err['msg']}" for err in error.errors())
    else:
        message = str(error)
    return MealPlannerError(f"ERROR in {function_name}()\n"
                            f"  Type:       {type(error).__name__}\n"
                            f"  Message:    {message}\n"
                            f"  Location:   {location}\n"
                            f"  Suggestion: {suggestion}")

In [28]:
def text_or_default(value, default):
    # Helper: an empty textbox, only spaces, or None becomes the default
    if value is None or str(value).strip() == "":
        return default
    return value


def format_ingredients(recipe):
    # Helper: "200 g pasta, 1 can tomatoes, ..." for the email AND the screen
    return ", ".join(f"{ing['quantity']} {ing['name']}" for ing in recipe["ingredients"])


def extract_json_text(raw_text):
    # Helper: keep only the {...} part of a model answer (fences, a sentence before it, or nothing at all)
    start, end = raw_text.find("{"), raw_text.rfind("}")
    if start == -1 or end < start:
        raise make_error("extract_json_text", ValueError("no JSON object in the model answer"),
                         f"model output, first 80 characters: {raw_text[:80]!r}",
                         "Click Generate again; if it keeps happening, check that the prompt still asks for JSON.")
    return raw_text[start:end + 1]


def parse_json_text(text, source, suggestion="Fix the JSON at that spot: usually a missing comma, bracket or quote."):
    # Helper: JSON text -> Python data; a syntax error reports line, column and the text around it
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        near = e.doc[max(0, e.pos - 30):e.pos + 30]
        raise make_error("parse_json_text", e, f"{source}, line {e.lineno}, column {e.colno}, near {near!r}",
                         suggestion) from e


def make_result(success, meal_plan, error, start_time):
    # Helper: success and failure results always have the same 4 keys
    return {"success": success, "meal_plan": meal_plan, "error": error,
            "generation_time": round(time.time() - start_time, 2)}


TRACE = []   # execution trace of the current run


def trace(step, detail):
    # Helper: print one timestamped line of the execution trace and keep it for lab_proof.md
    line = f"{datetime.now():%H:%M:%S} | {step:<18} | {detail}"
    TRACE.append(line)
    print(line)

### Test each helper on its own

Hand-made data only: no API call, no cost. `make_test_plan()` builds a fake meal plan with any number of recipes and ingredients; Step 5 reuses it to break the schema on purpose.

In [29]:
def make_test_plan(recipes=RECIPES_PER_PLAN, ingredients=INGREDIENTS_PER_RECIPE):
    # Hand-made test data shaped like a model answer
    return {"dietary_preference": "vegan",
            "recipes": [{"day": DAYS[i % len(DAYS)], "theme": f"Theme {i}", "dish_name": f"Dish {i}",
                         "ingredients": [{"name": f"item {j}", "quantity": "100 g"} for j in range(ingredients)],
                         "instructions": "Cook everything together."} for i in range(recipes)],
            "shopping_list": [{"name": "item 0", "total_quantity": "700 g", "used_in": ["Dish 0"]}]}


checks = [
    ("text_or_default('', 'random')", text_or_default("", "random"), "random"),
    ("text_or_default(None, 'random')", text_or_default(None, "random"), "random"),   # the original crashed on None
    ("text_or_default('Italian', 'random')", text_or_default("Italian", "random"), "Italian"),
    ("format_ingredients(test recipe)", format_ingredients(make_test_plan()["recipes"][0]),
     "100 g item 0, 100 g item 1, 100 g item 2, 100 g item 3, 100 g item 4"),
    ("extract_json_text(sentence + ```JSON fence)",
     extract_json_text('Sure! Here it is:\n```JSON\n{"a": 1}\n```'), '{"a": 1}'),
    ("parse_json_text('{\"a\": 1}')", parse_json_text('{"a": 1}', "test"), {"a": 1}),
    ("make_result: success keys == failure keys",
     sorted(make_result(True, {}, None, time.time())) == sorted(make_result(False, None, "x", time.time())), True),
]
for label, actual, expected in checks:
    print(f"{'PASS' if actual == expected else 'FAIL'}  {label:<44} -> {actual!r}")

PASS  text_or_default('', 'random')                -> 'random'
PASS  text_or_default(None, 'random')              -> 'random'
PASS  text_or_default('Italian', 'random')         -> 'Italian'
PASS  format_ingredients(test recipe)              -> '100 g item 0, 100 g item 1, 100 g item 2, 100 g item 3, 100 g item 4'
PASS  extract_json_text(sentence + ```JSON fence)  -> '{"a": 1}'
PASS  parse_json_text('{"a": 1}')                  -> {'a': 1}
PASS  make_result: success keys == failure keys    -> True


In [30]:
# What every error in this notebook now looks like, and what one trace line looks like
print(make_error("load_json_file", FileNotFoundError("[Errno 2] No such file or directory: 'x.json'"),
                 "file x.json", "Check the file name and the folder."))
print()
trace("example_step", "OK  one line of the execution trace")
TRACE.clear()

ERROR in load_json_file()
  Type:       FileNotFoundError
  Message:    [Errno 2] No such file or directory: 'x.json'
  Location:   file x.json
  Suggestion: Check the file name and the folder.

14:04:26 | example_step       | OK  one line of the execution trace


## Step 4: Modularize the functions

The "and" test: describe each original function in one sentence. Every "and" is a job that deserves its own function.

| Original function | One sentence | "and"s | Refactored into |
|---|---|---|---|
| `generate()` | builds the chain **and** fills defaults **and** calls the API **and** strips fences **and** parses JSON **and** validates **and** times the run | 6 | `call_meal_plan_api` → `extract_json_text` → `parse_json_text` → `validate_data`, called in order by `create_meal_plan()` |
| `send_meal_plan_email()` | formats the plan **and** builds the message **and** logs in to Gmail **and** sends | 3 | `format_meal_plan_email()`, `build_email_message()`, `send_email_message()` |
| `generate_plan_streaming()` | applies defaults **and** generates **and** formats **and** streams **and** validates again | 4 | calls `generate_meal_plan()` and the formatters; it only streams and keeps the plan |
| `send_email_ui()` | checks the address **and** generates a new plan **and** sends it | 2 | checks, then sends the plan already on screen |
| `run_basic_validation()` | re-checks rules the Pydantic models already enforce | 0, but a duplicate | removed: `MealPlan` is the only rule book |
| test cell | holds the test data **and** loops **and** calls the API **and** prints | 3 | `data/*.json` + `load_json_file()` + `run_workflow()` |

After the split, each function has one job:

| Job | Functions |
|---|---|
| Loading | `load_json_file` |
| Validation | `validate_data` (one function for all three Pydantic models) |
| API | `call_meal_plan_api` |
| Processing | `extract_json_text`, `parse_json_text`, `build_output_record`, the `format_*` functions, `build_email_message` |
| Saving / sending | `save_json_file`, `send_email_message` |
| Orchestration (only calls the others, in order) | `create_meal_plan`, `generate_meal_plan`, `run_workflow`, the two Gradio handlers |

### Data models: the input and the output schema

`MealRequest` is new: it validates the input (from the file or the form) and applies the "blank means default" rule in one place. The meal-plan models are the original ones with the counts taken from the constants. Their docstrings stay because Pydantic puts them into the JSON schema that is sent to the model.

In [31]:
class DietaryPreference(str, Enum):
    OMNIVORE = "omnivore"
    VEGETARIAN = "vegetarian"
    VEGAN = "vegan"
    KETO = "keto"


DIET_CHOICES = [diet.value for diet in DietaryPreference]   # the dropdown uses this list too


class MealRequest(BaseModel):
    # INPUT: one meal-plan request, from the JSON file or from the Gradio form
    dietary_preference: DietaryPreference
    theme_requests: str = DEFAULT_THEMES
    additional_preferences: str = DEFAULT_PREFERENCES

    @field_validator("theme_requests", "additional_preferences", mode="before")
    @classmethod
    def blank_means_default(cls, value, info):
        return text_or_default(value, cls.model_fields[info.field_name].default)


class RequestFile(BaseModel):
    # INPUT FILE: {"requests": [ {...}, {...} ]}
    requests: List[MealRequest] = Field(..., min_length=1)


class Ingredient(BaseModel):
    """Single ingredient with quantity."""
    name: str = Field(..., description="Name of the ingredient")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")


class Recipe(BaseModel):
    """A single themed dinner recipe."""
    day: str = Field(..., description="Day of the week")
    theme: str = Field(..., description="Cuisine theme (e.g., 'Italian', 'Mexican')")
    dish_name: str = Field(..., description="Name of the dish")
    ingredients: List[Ingredient] = Field(..., description=f"List of {INGREDIENTS_PER_RECIPE} ingredients")
    instructions: str = Field(..., description="Brief cooking instructions")

    @field_validator("ingredients")
    @classmethod
    def validate_ingredient_count(cls, v):
        if len(v) != INGREDIENTS_PER_RECIPE:
            raise ValueError(f"Recipe must have exactly {INGREDIENTS_PER_RECIPE} ingredients, got {len(v)}")
        return v


class ShoppingItem(BaseModel):
    """Consolidated shopping list item."""
    name: str = Field(..., description="Ingredient name")
    total_quantity: str = Field(..., description="Total quantity needed")
    used_in: List[str] = Field(..., description="List of dishes using this ingredient")


class MealPlan(BaseModel):
    """Complete weekly meal plan."""
    dietary_preference: str = Field(..., description="Selected dietary preference")
    recipes: List[Recipe] = Field(..., description=f"List of {RECIPES_PER_PLAN} recipes for the week")
    shopping_list: List[ShoppingItem] = Field(..., description="Consolidated shopping list")

    @field_validator("recipes")
    @classmethod
    def validate_recipe_count(cls, v):
        if len(v) != RECIPES_PER_PLAN:
            raise ValueError(f"Meal plan must have exactly {RECIPES_PER_PLAN} recipes, got {len(v)}")
        return v

    @field_validator("recipes")
    @classmethod
    def validate_unique_themes(cls, v):
        themes = [r.theme.lower() for r in v]
        if len(themes) != len(set(themes)):
            raise ValueError(f"All {RECIPES_PER_PLAN} themes must be unique")
        return v

### Prompt

Same wording as the original, so the model receives exactly the same instructions (checked in Step 6). The counts, the basics list and the days now come from the constants. The unused shopping-list prompt and parser are gone.

In [32]:
PLAN_FORMAT = JsonOutputParser(pydantic_object=MealPlan).get_format_instructions()

MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a professional meal planner. Generate a weekly meal plan with EXACTLY {recipe_count} themed dinners.

STRICT RULES - FOLLOW EXACTLY:
1. Each recipe MUST have EXACTLY {ingredient_count} main ingredients (do NOT count {excluded_basics})
2. ALL {recipe_count} THEMES MUST BE COMPLETELY DIFFERENT - no duplicates allowed! Use: Italian, Mexican, Japanese, Indian, Thai, Greek, American, Chinese, French, Korean, Mediterranean, Middle Eastern, etc.
3. All ingredients MUST be real, commonly available items
4. Follow the dietary preference strictly
5. Provide realistic quantities for 2 servings

Dietary Guidelines:
- Omnivore: Any ingredients allowed
- Vegetarian: No meat or fish, eggs and dairy OK
- Vegan: No animal products at all
- Keto: Low carb, high fat, no grains/sugar/starchy vegetables

Output valid JSON matching this exact structure:
{format_instructions}"""),
    ("human", """Create a weekly meal plan:
- Diet: {dietary_preference}
- Theme requests: {theme_requests}
- Preferences: {additional_preferences}

IMPORTANT: Each day MUST have a DIFFERENT cuisine theme. Generate {recipe_count} unique themed dinners for {days}.""")
]).partial(recipe_count=str(RECIPES_PER_PLAN),
           ingredient_count=str(INGREDIENTS_PER_RECIPE),
           excluded_basics=", ".join(EXCLUDED_BASICS[:-1]) + ", or " + EXCLUDED_BASICS[-1],
           days=f"{DAYS[0]}-{DAYS[-1]}",
           format_instructions=PLAN_FORMAT)

### Loading, validation and saving

Each function does one job and handles the errors that belong to that job: file errors in loading and saving, Pydantic errors in validation. `validate_data()` serves all three models (input file, form, model answer), so there is a single place that checks rules.

In [33]:
REQUEST_HINT = f"dietary_preference must be one of: {', '.join(DIET_CHOICES)}; themes and preferences must be text."
FILE_HINT = f"Fix the listed requests in the file (requests.0 is the first one). {REQUEST_HINT}"
PLAN_HINT = (f"The model broke a meal-plan rule ({RECIPES_PER_PLAN} recipes, {INGREDIENTS_PER_RECIPE} ingredients each, "
             "unique themes): click Generate again.")


def load_json_file(path):
    # Loading only: read a JSON file from disk and return its data
    try:
        text = Path(path).read_text(encoding="utf-8")
    except FileNotFoundError as e:
        raise make_error("load_json_file", e, f"file {Path(path).resolve()}",
                         "Check the file name and that the file is in the data/ folder next to this notebook.") from e
    except OSError as e:
        raise make_error("load_json_file", e, f"file {Path(path).resolve()}",
                         "Check that the file is not open in another program and that you can read it.") from e
    return parse_json_text(text, source=path)


def validate_data(model, data, source, suggestion):
    # Validation only: raw data -> Pydantic model; on failure every invalid field is listed
    try:
        return model.model_validate(data)
    except ValidationError as e:
        raise make_error("validate_data", e, f"{source} (checked against {model.__name__})", suggestion) from e


def save_json_file(data, path):
    # Saving only: write data as readable JSON, creating the folder if needed
    try:
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        Path(path).write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    except OSError as e:
        raise make_error("save_json_file", e, f"file {Path(path).resolve()}",
                         "Check that the folder is writable and the file is not open in another program.") from e
    return path

### The API call

The only function that talks to OpenAI. The call itself now sits inside `try` (in the original it was outside, so a wrong key crashed the app), and each OpenAI exception class gets its own message. `api_key` and `base_url` are only passed by the error tests in Step 5; normally the key comes from `.env`.

In [34]:
def root_cause(error):
    # Helper: follow the chain of wrapped errors down to the original one (e.g. the real network error)
    while error.__cause__ or error.__context__:
        error = error.__cause__ or error.__context__
    return error


def call_meal_plan_api(request, api_key=None, base_url=None):
    # API only: send the prompt for one validated request to OpenAI and return the raw text answer
    where = f"OpenAI chat API, model {OPENAI_MODEL}"
    if api_key is None:
        api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise make_error("call_meal_plan_api", ValueError("OPENAI_API_KEY is not set"), ".env file next to this notebook",
                         "Copy .env.example to .env and paste your OpenAI API key.")
    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE, timeout=TIMEOUT_SECONDS,
                     max_retries=API_MAX_RETRIES, api_key=api_key, base_url=base_url)
    try:
        return (MEAL_PLAN_PROMPT | llm).invoke(request.model_dump(mode="json")).content
    except openai.AuthenticationError as e:
        raise make_error("call_meal_plan_api", e, f"{where}, HTTP {e.status_code}",
                         "Check OPENAI_API_KEY in .env: the key is wrong, expired or revoked.") from e
    except openai.RateLimitError as e:
        raise make_error("call_meal_plan_api", e, f"{where}, HTTP {e.status_code}",
                         "Too many requests or no credit left: wait a minute, then check billing on platform.openai.com.") from e
    except openai.APITimeoutError as e:
        raise make_error("call_meal_plan_api", e, f"{where}, no answer within {TIMEOUT_SECONDS}s",
                         "Try again; if it keeps happening, raise TIMEOUT_SECONDS.") from e
    except openai.APIConnectionError as e:
        cause = root_cause(e)
        raise make_error("call_meal_plan_api", e, f"{where}, network detail: {type(cause).__name__}: {cause}",
                         "Check your internet connection, VPN or proxy, then try again.") from e
    except openai.APIError as e:
        raise make_error("call_meal_plan_api", e, f"{where}, HTTP {getattr(e, 'status_code', 'unknown')}",
                         "Look up this status code on the OpenAI error-codes page.") from e

### Orchestrators: they only call the other functions, in order

- `create_meal_plan()` is the transformation: request → API → JSON → validated plan. Both entry points below use it, so the app and the file workflow cannot drift apart.
- `generate_meal_plan()` is the app's entry point: form values in, result out.
- `run_workflow()` is the file workflow for the proof: input file in, output record out. Invalid input stops it **before** the paid API call.

Both entry points turn a `MealPlannerError` into the same result shape, so nothing fails silently and nothing crashes the app. Unexpected bugs are not caught and still show a full traceback.

In [35]:
def create_meal_plan(request):
    # Transformation: validated request -> validated MealPlan, one function call per step
    trace("call_meal_plan_api", f"sending request to {OPENAI_MODEL} ...")
    raw_text = call_meal_plan_api(request)
    trace("call_meal_plan_api", f"OK  {len(raw_text)} characters received")
    data = parse_json_text(extract_json_text(raw_text), "model output", "The model sent broken JSON: click Generate again.")
    trace("parse_json_text", "OK  the answer is valid JSON")
    plan = validate_data(MealPlan, data, "model output", PLAN_HINT)
    trace("validate_data", f"OK  {len(plan.recipes)} recipes x {INGREDIENTS_PER_RECIPE} ingredients, unique themes")
    return plan


def generate_meal_plan(request_data, source="Gradio form"):
    # App entry point: form values -> result dict (known errors come back as a message, never a crash)
    TRACE.clear()
    start = time.time()
    try:
        request = validate_data(MealRequest, request_data, source, REQUEST_HINT)
        return make_result(True, create_meal_plan(request).model_dump(), None, start)
    except MealPlannerError as e:
        return make_result(False, None, str(e), start)


def build_output_record(request, result):
    # Processing only: input payload + result + run facts = the record a reviewer inspects
    return {"created_at": datetime.now().isoformat(timespec="seconds"), "model": OPENAI_MODEL,
            "input": request.model_dump(mode="json"), **result}


def run_workflow(requests_file, request_number):
    # File workflow (trigger -> transform -> action): input file -> validated request -> plan -> saved record
    TRACE.clear()
    start = time.time()
    trace("run_workflow", f"started: {requests_file}, request #{request_number}")
    try:
        data = load_json_file(requests_file)
        trace("load_json_file", f"OK  {requests_file} read")
        requests = validate_data(RequestFile, data, requests_file, FILE_HINT).requests
        request = requests[request_number - 1]
        trace("validate_data", f"OK  {len(requests)} valid requests, using #{request_number}: {request.model_dump(mode='json')}")
        result = make_result(True, create_meal_plan(request).model_dump(), None, start)
        save_json_file(build_output_record(request, result), OUTPUT_FILE)
        trace("save_json_file", f"OK  output record written to {OUTPUT_FILE}")
        return result
    except MealPlannerError as e:
        trace("run_workflow", "STOPPED, error below")
        return make_result(False, None, str(e), start)

### Output formatting, email and the app handlers

The formatters keep the original layout; the ingredient line now comes from `format_ingredients()`. Building the email and sending it are two functions, so the message can be tested offline. The **Email Plan** handler now receives the plan that is on screen (stored in `gr.State`) instead of generating a new one.

In [36]:
def format_meal_plan_email(meal_plan):
    # Processing only: plain-text email body, shopping list first (same layout as the original)
    lines = ["=" * 40, f"WEEKLY MEAL PLAN ({meal_plan['dietary_preference'].upper()})", "=" * 40, "",
             "SHOPPING LIST", "-" * 20]
    lines += [f"- {item['total_quantity']} {item['name']}" for item in meal_plan["shopping_list"]]
    lines += ["", "=" * 40, "RECIPES", "=" * 40]
    for recipe in meal_plan["recipes"]:
        lines += [f"\n{recipe['day'].upper()} - {recipe['theme']}", recipe["dish_name"],
                  f"Ingredients: {format_ingredients(recipe)}", f"Instructions: {recipe['instructions']}"]
    return "\n".join(lines)


def format_shopping_list(meal_plan):
    # Processing only: shopping list as Markdown for the screen
    items = [f"- {item['total_quantity']} **{item['name']}**" for item in meal_plan["shopping_list"]]
    return "\n".join(["## Shopping List\n"] + items)


def format_single_recipe(recipe):
    # Processing only: one recipe as compact Markdown for the screen
    return f"**{recipe['day']}** | {recipe['theme']} | *{recipe['dish_name']}*\n{format_ingredients(recipe)}\n"


def build_email_message(meal_plan, sender, recipient):
    # Processing only: build the email (no network, so it can be tested offline)
    message = EmailMessage()
    message["From"], message["To"], message["Subject"] = sender, recipient, EMAIL_SUBJECT
    message.set_content(format_meal_plan_email(meal_plan))
    return message


def send_email_message(message, app_password):
    # Sending only: log in to Gmail and send an already-built message
    where = f"SMTP server {SMTP_HOST}:{SMTP_PORT}, sender {message['From']}"
    try:
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=TIMEOUT_SECONDS) as server:
            server.starttls()
            server.login(message["From"], app_password)
            server.send_message(message)
    except smtplib.SMTPAuthenticationError as e:
        raise make_error("send_email_message", e, where,
                         "Use a Gmail App Password (not your normal password) as EMAIL_APP_PASSWORD in .env.") from e
    except (smtplib.SMTPException, OSError) as e:
        raise make_error("send_email_message", e, where,
                         "Check the recipient address, your internet connection and the SMTP settings.") from e

In [37]:
def generate_plan_streaming(dietary_pref, theme_requests, additional_prefs):
    # Gradio "Generate": ONE API call; the plan streams to the screen and is kept in gr.State for "Email Plan"
    yield "Generating your meal plan...", None
    result = generate_meal_plan({"dietary_preference": dietary_pref, "theme_requests": theme_requests,
                                 "additional_preferences": additional_prefs})
    if not result["success"]:
        yield f"**Could not create the meal plan**\n```\n{result['error']}\n```", None
        return
    plan = result["meal_plan"]
    output = f"## Weekly {plan['dietary_preference'].title()} Meal Plan\n\n{format_shopping_list(plan)}\n\n---\n\n## Recipes\n\n"
    for recipe in plan["recipes"]:
        output += format_single_recipe(recipe) + "\n"
        yield output, plan
        time.sleep(0.05)
    yield output + f"---\n*Generated in {result['generation_time']}s | All checks passed*", plan


def send_email_ui(email, meal_plan):
    # Gradio "Email Plan": sends the plan that is ON SCREEN (no second API call, same recipes)
    if not meal_plan:
        return "Click Generate first: there is no plan on screen yet."
    if not email or "@" not in email:
        return "Enter a valid email address."
    if not (EMAIL_SENDER and EMAIL_APP_PASSWORD):
        return "Email is not set up: add EMAIL_SENDER and EMAIL_APP_PASSWORD to .env (see .env.example)."
    try:
        send_email_message(build_email_message(meal_plan, EMAIL_SENDER, email), EMAIL_APP_PASSWORD)
        return f"Email sent to {email}"
    except MealPlannerError as e:
        return str(e)

### Checkpoint: parsing no longer needs an API call

In the original, JSON parsing and validation were stuck inside `generate()`, behind the API call. Now a hand-made model answer (a sentence plus a fenced JSON block) goes through the same steps offline.

In [38]:
fake_answer = "Here is your plan:\n```json\n" + json.dumps(make_test_plan()) + "\n```"
plan = validate_data(MealPlan, parse_json_text(extract_json_text(fake_answer), "hand-made answer"), "hand-made answer", PLAN_HINT)
print(f"Parsed and validated offline: {len(plan.recipes)} recipes, {len(plan.recipes[0].ingredients)} ingredients each\n")
print(format_single_recipe(plan.model_dump()["recipes"][0]))

Parsed and validated offline: 7 recipes, 5 ingredients each

**Monday** | Theme 0 | *Dish 0*
100 g item 0, 100 g item 1, 100 g item 2, 100 g item 3, 100 g item 4



## Step 5: Add error handling

The rule I followed for every `except`:

- The function where something breaks **raises** a `MealPlannerError` built by `make_error()`: function name, error type, message, location and a suggestion.
- The entry points (`generate_meal_plan`, `run_workflow`, `send_email_ui`) **return** that message as a result the app can show. Nothing prints an error and then carries on silently.
- No bare `except:` (it would also swallow Ctrl+C). Only named exception types are caught, so a real bug still shows its traceback.

| Where errors can occur | Function | Exceptions handled | Location shown |
|---|---|---|---|
| File operations | `load_json_file`, `save_json_file` | `FileNotFoundError`, `OSError` | full file path |
| JSON parsing | `parse_json_text` | `json.JSONDecodeError` | line, column and the text around it |
| Data validation | `validate_data` | Pydantic `ValidationError` | each invalid field path and why |
| API calls | `call_meal_plan_api` | `AuthenticationError`, `RateLimitError`, `APITimeoutError`, `APIConnectionError`, `APIError` | model, HTTP status, network detail |
| Response processing | `extract_json_text`, `validate_data(MealPlan)` | no JSON in the answer, missing fields, wrong counts | start of the answer, field path |
| Email | `send_email_message` | `SMTPAuthenticationError`, `SMTPException`, `OSError` | SMTP server and sender |

Below, each one is broken on purpose. `EVIDENCE` collects the messages for `lab_proof.md`.

In [39]:
EVIDENCE = {}   # error messages collected for lab_proof.md

### 5.1 Broken JSON from the model

Original: `JSON parsing error: Expecting value: line 1 column 14 (char 13)`, with no source and no suggestion. An answer with no JSON at all gave an equally vague `Expecting value: line 1 column 1 (char 0)`.

In [40]:
for answer in ['{"recipes": [}', "Sorry, I cannot create that meal plan."]:
    try:
        parse_json_text(extract_json_text(answer), "model output", "The model sent broken JSON: click Generate again.")
    except MealPlannerError as e:
        print(e, "\n")

ERROR in parse_json_text()
  Type:       JSONDecodeError
  Message:    Expecting value: line 1 column 14 (char 13)
  Location:   model output, line 1, column 14, near '{"recipes": [}'
  Suggestion: The model sent broken JSON: click Generate again. 

ERROR in extract_json_text()
  Type:       ValueError
  Message:    no JSON object in the model answer
  Location:   model output, first 80 characters: 'Sorry, I cannot create that meal plan.'
  Suggestion: Click Generate again; if it keeps happening, check that the prompt still asks for JSON. 



### 5.2 A meal plan that breaks the schema: 6 recipes, then one recipe with 4 ingredients

Original: Pydantic's raw error text inside `str(e)`, which also dumps the input it received, with no hint about what to do.

In [41]:
six_recipes = make_test_plan(recipes=6)
four_ingredients = make_test_plan()
four_ingredients["recipes"][3]["ingredients"].pop()   # Thursday loses one ingredient

for label, bad_plan in [("6 recipes", six_recipes), ("Thursday has 4 ingredients", four_ingredients)]:
    try:
        validate_data(MealPlan, bad_plan, f"hand-made plan ({label})", PLAN_HINT)
    except MealPlannerError as e:
        print(e, "\n")

ERROR in validate_data()
  Type:       ValidationError
  Message:    1 invalid field(s)
              - recipes: Value error, Meal plan must have exactly 7 recipes, got 6
  Location:   hand-made plan (6 recipes) (checked against MealPlan)
  Suggestion: The model broke a meal-plan rule (7 recipes, 5 ingredients each, unique themes): click Generate again. 

ERROR in validate_data()
  Type:       ValidationError
  Message:    1 invalid field(s)
              - recipes.3.ingredients: Value error, Recipe must have exactly 5 ingredients, got 4
  Location:   hand-made plan (Thursday has 4 ingredients) (checked against MealPlan)
  Suggestion: The model broke a meal-plan rule (7 recipes, 5 ingredients each, unique themes): click Generate again. 



### 5.3 Empty textbox, then `None`

Original: an empty box worked, but `None` crashed both Gradio handlers with `AttributeError: 'NoneType' object has no attribute 'strip'`. Now both become the default inside `MealRequest`.

In [42]:
for themes in ["", "   ", None]:
    request = validate_data(MealRequest, {"dietary_preference": "vegan", "theme_requests": themes}, "form test", REQUEST_HINT)
    print(f"theme_requests={themes!r:<6} -> {request.theme_requests!r}")

theme_requests=''     -> 'random diverse cuisines'
theme_requests='   '  -> 'random diverse cuisines'
theme_requests=None   -> 'random diverse cuisines'


### 5.4 OpenAI key missing, then a fake key · network call, no cost

Original: a missing key crashed the env cell with a `TypeError`; a wrong key raised an unhandled `AuthenticationError` (the call sat outside `try`), so the app showed a generic error and the test loop stopped. OpenAI rejects the fake key before any tokens are used, so this test costs nothing.

In [43]:
test_request = MealRequest(dietary_preference="vegan")
for label, key in [("API error: missing key", ""), ("API error: wrong key", "sk-fake-key-for-testing-only")]:
    try:
        call_meal_plan_api(test_request, api_key=key)
    except MealPlannerError as e:
        EVIDENCE[label] = str(e)
        print(e, "\n")

ERROR in call_meal_plan_api()
  Type:       ValueError
  Message:    OPENAI_API_KEY is not set
  Location:   .env file next to this notebook
  Suggestion: Copy .env.example to .env and paste your OpenAI API key. 

ERROR in call_meal_plan_api()
  Type:       OpenAIAuthenticationError
  Message:    Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-fake-****************only. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
  Location:   OpenAI chat API, model gpt-4o-mini, HTTP 401
  Suggestion: Check OPENAI_API_KEY in .env: the key is wrong, expired or revoked. 



### 5.5 Network down

Instead of turning off Wi-Fi, the client is pointed at a closed port on this computer, which fails the same way a dropped network does: no connection can be made. No cost; the built-in retries make it take a few seconds.

Original: an unhandled `APIConnectionError: Connection error.` with no detail.

In [44]:
try:
    call_meal_plan_api(test_request, api_key="sk-fake-key-for-testing-only", base_url="http://127.0.0.1:9/v1")
except MealPlannerError as e:
    EVIDENCE["Network error: server unreachable"] = str(e)
    print(e)

ERROR in call_meal_plan_api()
  Type:       OpenAIConnectionError
  Message:    Connection error.
  Location:   OpenAI chat API, model gpt-4o-mini, network detail: ConnectionRefusedError: [Errno 61] Connection refused
  Suggestion: Check your internet connection, VPN or proxy, then try again.


### 5.6 Email with a wrong app password · network call, nothing is sent

Original: it generated a second plan first, then showed `Failed to send email: ...`. Now no plan is generated, and the login failure says what to fix. The login fails before anything is sent.

In [45]:
message = build_email_message(make_test_plan(), "nobody@example.com", "you@example.com")
try:
    send_email_message(message, app_password="wrong-app-password")
except MealPlannerError as e:
    print(e)

ERROR in send_email_message()
  Type:       SMTPServerDisconnected
  Message:    Connection unexpectedly closed
  Location:   SMTP server smtp.gmail.com:587, sender nobody@example.com
  Suggestion: Check the recipient address, your internet connection and the SMTP settings.


## Step 6: Test the refactored code

Valid input must behave like the Step 1 baseline, and every broken input must give a clear message. Only 6.1 calls the paid API.

### 6.1 Valid data: the full workflow · Real API call

Request #2 from `data/meal_requests.json` → OpenAI → validated plan → `output/meal_plan_record.json`. The trace printed here is the execution trace in `lab_proof.md`.

In [46]:
result = run_workflow(REQUESTS_FILE, PROOF_REQUEST)
PROOF_TRACE = list(TRACE)   # keep this run's trace; later tests start new traces

if result["success"]:
    print(f"\nSUCCESS in {result['generation_time']}s: {len(result['meal_plan']['recipes'])} recipes, "
          f"{len(result['meal_plan']['shopping_list'])} shopping-list items")
    for recipe in result["meal_plan"]["recipes"]:
        print(f"  {recipe['day']:<10} {recipe['theme']:<18} {recipe['dish_name']}")
else:
    print("\n" + result["error"])

14:04:32 | run_workflow       | started: data/meal_requests.json, request #2
14:04:32 | load_json_file     | OK  data/meal_requests.json read
14:04:32 | validate_data      | OK  4 valid requests, using #2: {'dietary_preference': 'vegetarian', 'theme_requests': 'Italian, Asian, Mexican', 'additional_preferences': 'no mushrooms'}
14:04:32 | call_meal_plan_api | sending request to gpt-4o-mini ...
14:04:33 | run_workflow       | STOPPED, error below

ERROR in call_meal_plan_api()
  Type:       OpenAIAuthenticationError
  Message:    Error code: 401 - {'error': {'message': 'Your API key has been invalidated.', 'type': None, 'code': 'token_invalidated', 'param': None}, 'status': 401}
  Location:   OpenAI chat API, model gpt-4o-mini, HTTP 401
  Suggestion: Check OPENAI_API_KEY in .env: the key is wrong, expired or revoked.


### 6.2 Compare with the baseline

Same input, original code vs refactored code. The dishes differ between any two runs (temperature 0.7), so the comparison is about structure: same result keys, 7 recipes, 5 ingredients each, 7 different themes.

In [47]:
def describe_run(run):
    # Structure of one run's result, to compare original vs refactored
    plan = run.get("meal_plan") or {"recipes": [], "shopping_list": []}
    return {"success": run["success"],
            "recipes": len(plan["recipes"]),
            "ingredients per recipe": sorted({len(r["ingredients"]) for r in plan["recipes"]}),
            "different themes": len({r["theme"].lower() for r in plan["recipes"]}),
            "shopping-list items": len(plan["shopping_list"]),
            "result keys": ", ".join(sorted(run)),
            "seconds": run.get("generation_time")}


before = describe_run(json.loads(Path(BASELINE_FILE).read_text(encoding="utf-8")))
after = describe_run(result)
print(f"{'':<24}{'original':<48}refactored")
for key in after:
    print(f"{key:<24}{str(before[key]):<48}{after[key]}")

                        original                                        refactored
success                 False                                           False
recipes                 0                                               0
ingredients per recipe  []                                              []
different themes        0                                               0
shopping-list items     0                                               0
result keys             error, meal_plan, success                       error, generation_time, meal_plan, success
seconds                 None                                            0.6


### 6.3 Same prompt and same email text as the original (offline)

The original's config, models, prompt and email cells are loaded again (no API key needed), and their output is compared with the refactored code for the same input.

In [48]:
original_ns = {"__name__": "original_notebook"}
for code in [original_cells[i] for i in (1, 3, 4, 6)]:   # config, models, prompts, email
    exec(code, original_ns)

original_format = original_ns["JsonOutputParser"](pydantic_object=original_ns["MealPlan"]).get_format_instructions()
original_prompt = original_ns["MEAL_PLAN_PROMPT"].format(**baseline_input, format_instructions=original_format)
print("Prompt sent to the model is identical:", original_prompt == MEAL_PLAN_PROMPT.format(**baseline_input))
print("Email text is identical:              ",
      original_ns["format_meal_plan_email"](make_test_plan()) == format_meal_plan_email(make_test_plan()))

Prompt sent to the model is identical: True
Email text is identical:               True


### 6.4 Generate, then Email: is the emailed plan the plan on screen?

In the app the email is built from the same plan object the screen shows (`gr.State`), so every dish on screen must be in the email. The button also refuses to work before a plan exists, instead of generating one.

In [49]:
shown_plan = result["meal_plan"] if result["success"] else make_test_plan()
screen_text = "".join(format_single_recipe(recipe) for recipe in shown_plan["recipes"])
email_text = build_email_message(shown_plan, "me@example.com", "you@example.com").get_content()
print("Every dish on screen is in the email:",
      all(r["dish_name"] in screen_text and r["dish_name"] in email_text for r in shown_plan["recipes"]))
print("Email Plan before Generate     ->", send_email_ui("you@example.com", None))
print("Email Plan with a bad address  ->", send_email_ui("not-an-email", shown_plan))

Every dish on screen is in the email: True
Email Plan before Generate     -> Click Generate first: there is no plan on screen yet.
Email Plan with a bad address  -> Enter a valid email address.


### 6.5 Invalid data: missing file, malformed JSON, invalid requests

This closes the "file-input gap" from the hints: the original read its input from a form and an inline `TEST_CASES` list, so there was no file to break. All three runs stop before the API call.

| File | What is wrong | Expected message |
|---|---|---|
| `data/does_not_exist.json` | does not exist | full path and where to put the file |
| `data/malformed.json` | missing comma between two requests | line and column of the error |
| `data/invalid_requests.json` | unknown diet `paleo`, missing `dietary_preference`, a number and a list where text is expected | each invalid field and why |

In [50]:
for label, path in [("Missing file", "data/does_not_exist.json"),
                    ("Malformed JSON", "data/malformed.json"),
                    ("Invalid data", "data/invalid_requests.json")]:
    print(f"--- {label}: run_workflow('{path}')")
    failed = run_workflow(path, 1)
    EVIDENCE[f"{label}: {path}"] = failed["error"]
    print(failed["error"], "\n")

--- Missing file: run_workflow('data/does_not_exist.json')
14:04:33 | run_workflow       | started: data/does_not_exist.json, request #1
14:04:33 | run_workflow       | STOPPED, error below
ERROR in load_json_file()
  Type:       FileNotFoundError
  Message:    [Errno 2] No such file or directory: 'data/does_not_exist.json'
  Location:   file /Users/macbook-air/Desktop/AI consulting/Week2/Day5/lab-python-low-code/data/does_not_exist.json
  Suggestion: Check the file name and that the file is in the data/ folder next to this notebook. 

--- Malformed JSON: run_workflow('data/malformed.json')
14:04:33 | run_workflow       | started: data/malformed.json, request #1
14:04:33 | run_workflow       | STOPPED, error below
ERROR in parse_json_text()
  Type:       JSONDecodeError
  Message:    Expecting ',' delimiter: line 4 column 5 (char 120)
  Location:   data/malformed.json, line 4, column 5, near 'nal_preferences": "none"}\n    {"dietary_preference": "keto",'
  Suggestion: Fix the JSON at t

### 6.6 Verify the output record

The reviewer's check: open the file in the destination and confirm that the input really drove the output.

In [51]:
VERIFY_CHECKS = {}
if result["success"]:
    saved = load_json_file(OUTPUT_FILE)
    payload, summary = saved["input"], describe_run(saved)
    VERIFY_CHECKS = {
        f"{OUTPUT_FILE} re-loads and equals the run result": saved["meal_plan"] == result["meal_plan"],
        "diet in the output equals the diet in the input":
            saved["meal_plan"]["dietary_preference"].lower() == payload["dietary_preference"],
        f"{RECIPES_PER_PLAN} recipes, {INGREDIENTS_PER_RECIPE} ingredients each, {RECIPES_PER_PLAN} different themes":
            (summary["recipes"], summary["ingredients per recipe"], summary["different themes"])
            == (RECIPES_PER_PLAN, [INGREDIENTS_PER_RECIPE], RECIPES_PER_PLAN),
    }
    if payload["additional_preferences"].lower().startswith("no "):   # simple check for a "no <ingredient>" wish
        banned = payload["additional_preferences"][3:].strip().lower().rstrip("s")
        names = [i["name"].lower() for r in saved["meal_plan"]["recipes"] for i in r["ingredients"]]
        VERIFY_CHECKS[f"preference '{payload['additional_preferences']}' respected"] = not any(banned in n for n in names)

for label, ok in VERIFY_CHECKS.items():
    print(f"{'PASS' if ok else 'FAIL'}  {label}")
if not VERIFY_CHECKS:
    print("Skipped: the run in 6.1 did not succeed (see its error message).")

Skipped: the run in 6.1 did not succeed (see its error message).


### 6.7 Compare behaviour: original vs refactored

| Break this | How | Original shows | Refactored shows |
|---|---|---|---|
| OpenAI key | missing, then fake (5.4) | `TypeError: str expected, not NoneType`; unhandled `AuthenticationError` | `call_meal_plan_api()`: key not set / HTTP 401, "check OPENAI_API_KEY in .env" |
| Network | unreachable server (5.5) | unhandled `APIConnectionError: Connection error.` | `APIConnectionError` plus the underlying network detail and what to check |
| JSON from the model | `'{"recipes": [}'` (5.1) | `JSON parsing error: Expecting value: line 1 column 14 (char 13)` | `parse_json_text()`: line, column, the text around it, "click Generate again" |
| Meal-plan schema | 6 recipes, 4 ingredients (5.2) | Pydantic's raw text inside `str(e)` | `validate_data()`: field path such as `recipes.3.ingredients`, the rule, a suggestion |
| Email | wrong app password (5.6) | a second plan is generated, then `Failed to send email: ...` | no extra plan; `SMTPAuthenticationError` and "use a Gmail App Password" |
| Empty textbox | `""`, then `None` (5.3) | `None` crashes `.strip()` | the default is applied in `MealRequest` |
| Input file | missing, malformed, invalid (6.5) | not testable: there was no input file | full path; line and column; every invalid field |

**Does it work the same?** Same prompt and email text (6.3) and the same plan structure as the baseline (6.2). **Are the errors better?** Every message now names the function, the error type, the location and a fix. **Is the code easier to understand?** The 40-line `generate()` that did seven jobs is now a 10-line `create_meal_plan()` that calls one-job functions in order, and parsing and validation can be tested without an API call.

## Proof of learning: fill `lab_proof.md`

`lab_proof.md` is the reviewer's entry point. Its explanations are written by hand; this cell fills its evidence blocks (run settings, execution trace, input payload, output record, checks, error messages) from this run. Running it again replaces them.

In [52]:
def fill_block(text, name, content):
    # Helper: replace whatever sits between <!-- name:start --> and <!-- name:end --> with fresh evidence
    start, end = f"<!-- {name}:start -->", f"<!-- {name}:end -->"
    before, rest = text.split(start)
    return f"{before}{start}\n{content}\n{end}{rest.split(end)[1]}"


def fenced(content, language=""):
    # Helper: wrap text in a Markdown code block
    return f"```{language}\n{content}\n```"


run_settings = {"input_file": REQUESTS_FILE, "request_number": PROOF_REQUEST, "output_file": OUTPUT_FILE,
                "model": OPENAI_MODEL, "temperature": TEMPERATURE, "timeout_seconds": TIMEOUT_SECONDS,
                "max_retries": API_MAX_RETRIES, "recipes_per_plan": RECIPES_PER_PLAN,
                "ingredients_per_recipe": INGREDIENTS_PER_RECIPE, "api_key": "read from .env, never written out"}
payload = load_json_file(REQUESTS_FILE)["requests"][PROOF_REQUEST - 1]
blocks = {
    "config": fenced(json.dumps(run_settings, indent=2), "json"),
    "trace": fenced("\n".join(PROOF_TRACE)),
    "input": fenced(json.dumps(payload, indent=2), "json"),
    "output": fenced(Path(OUTPUT_FILE).read_text(encoding="utf-8"), "json") if result["success"]
              else "The run failed, so there is no output record:\n\n" + fenced(result["error"]),
    "checks": "\n".join(f"- {'PASS' if ok else 'FAIL'}: {label}" for label, ok in VERIFY_CHECKS.items())
              or "- Not verified: the run failed (see sections 2 and 4).",
    "errors": "\n\n".join(f"**{label}**\n\n{fenced(message)}" for label, message in EVIDENCE.items()),
}

proof = Path("lab_proof.md").read_text(encoding="utf-8")
for name, content in blocks.items():
    proof = fill_block(proof, name, content)
Path("lab_proof.md").write_text(proof, encoding="utf-8")
print(f"lab_proof.md updated ({', '.join(blocks)}); workflow run succeeded: {result['success']}")

lab_proof.md updated (config, trace, input, output, checks, errors); workflow run succeeded: False


## The app

Same layout as the original. What changed: the dropdown uses `DIET_CHOICES`, the plan on screen is stored in `gr.State` so **Email Plan** sends exactly that plan, the status box is taller so a full error message fits, and the app runs locally only. The original used `share=True`, which published a public link to an app that spends your API credit and can send email from your account.

In [53]:
with gr.Blocks(title="Weekly Meal Planner") as app:
    gr.Markdown(f"# Weekly Meal Planner\nGenerate {RECIPES_PER_PLAN} themed dinners with a consolidated shopping list.")
    plan_state = gr.State(None)   # the plan on screen, reused by Email Plan

    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(choices=DIET_CHOICES, value="omnivore", label="Diet")
            theme_input = gr.Textbox(label="Themes (optional)", placeholder="Italian, Mexican...", lines=1)
            additional_input = gr.Textbox(label="Preferences (optional)", placeholder="no nuts, quick meals...", lines=1)
            generate_btn = gr.Button("Generate", variant="primary")

            gr.Markdown("---")
            email_input = gr.Textbox(label="Email", placeholder="you@example.com")
            send_btn = gr.Button("Email Plan")
            email_status = gr.Textbox(label="Status", interactive=False, lines=4)

        with gr.Column(scale=2):
            output_display = gr.Markdown("Click **Generate** to start.")

    generate_btn.click(fn=generate_plan_streaming, inputs=[dietary_dropdown, theme_input, additional_input],
                       outputs=[output_display, plan_state])
    send_btn.click(fn=send_email_ui, inputs=[email_input, plan_state], outputs=[email_status])

In [54]:
app.launch()   # local only (no share=True): http://127.0.0.1:7860

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


14:06:00 | call_meal_plan_api | sending request to gpt-4o-mini ...


## Verification criteria

- [x] Code is refactored into helper functions (Step 3)
- [x] Code is modular, with separate functions for separate concerns (Step 4)
- [x] Error handling shows WHERE errors occur, no silent failures (Step 5)
- [x] All error types are handled explicitly:
  - [x] `FileNotFoundError`: `load_json_file`, test 6.5
  - [x] `JSONDecodeError`: `parse_json_text`, tests 5.1 and 6.5
  - [x] Pydantic `ValidationError`: `validate_data`, tests 5.2 and 6.5
  - [x] OpenAI `APIError`: `call_meal_plan_api`, test 5.4
  - [x] Network errors: `APIConnectionError` and `APITimeoutError`, test 5.5
- [x] Code works with the provided JSON data (6.1, compared with the baseline in 6.2)
- [x] Error messages include the function name, the error type and message, the location and a helpful suggestion (`make_error`)
- [x] No secrets in the notebook; `.env` is listed in `.gitignore`

Steps 7 and 8 (API wrapper, logging) are optional and not part of this submission.